# SNA-MXH · E3C — Export Top-K · FIXED v2 (12 checkpoints + reuse E1)

> **v2 fix (06/11/2026):** Bỏ static assertion quá chặt về chuỗi `/content/drive/MyDrive/` trong group exporter. Mỗi lượt tạo một bản exporter tạm, thay root Drive cố định của project (nếu có) bằng staging root tương ứng. Sau đó kiểm tra Top-K, train-mask, và metric consistency. **Chưa chạy thực nghiệm trên Colab của nhóm**; dùng `MAX_NEW_EXPORTS = 1` để kiểm thử trước khi chạy toàn bộ.


**Mục đích:** Export Top-20 cho RLMRec-Con Full (`lightgcn_plus`) tại các `kd_weight = 0, 0.005, 0.01, 0.02, 0.05`, seeds `2023/2024/2025`. **Không train lại.**

Dựa trên notebook `04_E1_Export_TopK.ipynb` và E3C protocol của nhóm:

- `0.01`: dùng lại Top-K đã kiểm chứng từ E1, không inference lại.
- 12 run còn lại: dùng checkpoint E3C thực tế và `src/inference/export_topk.py` (exporter E1 của nhóm).
- Tránh ghi đè E1: tạo **staging root riêng trong `/content/`** để exporter E1 tìm thấy checkpoint, metrics thuộc **đúng run E3C**, sau đó chỉ chuyển output đã kiểm tra về E3C.
- Mask: **train-only**; Top-K=20; không mask validation; giữ định nghĩa evaluation nhất quán E1/E2/E3B.
- Có checkpoint SHA256, metadata provenance, kiểm tra 9860 test users, 9332 items, scores/duplicates/train-mask, trạng thái resume và CSV summary.

> **Giới hạn cần biết:** Vì file nguồn `src/inference/export_topk.py` chưa được đính kèm, phiên bản này **tái sử dụng nguyên exporter ở GitHub tại thời điểm Colab chạy** và kiểm tra hợp đồng đầu vào/đầu ra của nó. Nếu exporter đã thay đổi hoặc kiểm tra đường dẫn cố định ngoài `--root`, notebook sẽ dừng với thông báo lỗi — **không âm thầm dùng checkpoint E1**. Chưa xác nhận đã chạy end-to-end trên Colab.

**Chạy:** Google Colab → Runtime → Change runtime type → **GPU** → Run all. Mặc định tất cả run. Có thể đặt `MAX_NEW_EXPORTS=1` để smoke test một checkpoint trước.

In [ ]:
# CELL 1 — CONFIG & COLAB
from pathlib import Path
import os, sys, json, re, csv, math, ast, pickle, hashlib, shutil, subprocess, traceback
from datetime import datetime, timezone
import numpy as np
import scipy.sparse as sp
import torch
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA_DIR = ROOT / 'data/raw/amazon'
E3C_ROOT = ROOT / 'experiments/E3C'
PRED_ROOT = ROOT / 'experiments/predictions/lightgcn_plus'
MODEL, DATASET, K = 'lightgcn_plus', 'amazon', 20
WEIGHTS = ['0', '0.005', '0.01', '0.02', '0.05']
SEEDS = [2023, 2024, 2025]
REUSED_WEIGHT = '0.01'
RLMREC_COMMIT = '22413752246de3dee8ab0d509f7f7a8889080f95'

# Controls
MAX_NEW_EXPORTS = None  # None = export đủ 12; đặt 1 để smoke test trước
RESUME = True            # bỏ qua run đã xuất và verify đầy đủ
FORCE = False            # True = cho phép thay thế output run E3C cũ
STOP_ON_ERROR = True     # dừng nếu một run thất bại (không chạy tiếp sai hàng loạt)
COPY_E1_REFERENCE = True # copy riêng 3 Top-K weight=0.01 vào E3C
CHECK_TRAIN_MASK = True
MAX_METRIC_ABS_DIFF = 2e-4  # sai khác test Recall/NDCG so với trainer log

assert torch.cuda.is_available(), 'Please enable GPU in Colab (Runtime > Change runtime type).'
assert DATA_DIR.is_dir(), f'Missing data: {DATA_DIR}'
assert (DATA_DIR / 'trn_mat.pkl').is_file() and (DATA_DIR / 'tst_mat.pkl').is_file()
assert E3C_ROOT.is_dir(), f'Missing E3C: {E3C_ROOT}'
print('Python:', sys.version.split()[0], '| Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Experiment:', E3C_ROOT)
print('New exports:', 4 * len(SEEDS), '| E1 reuse:', len(SEEDS))

In [ ]:
# CELL 2 — REPOSITORIES (GIỮ ĐÚNG RLMRec COMMIT E1/E3C)
GROUP_REPO = Path('/content/project_sna_group14')
GROUP_REPO_URL = 'https://github.com/trungkien1502/project_sna_group14.git'
RLMREC_REPO = Path('/content/RLMRec')
RLMREC_URL = 'https://github.com/HKUDS/RLMRec.git'

def run_checked(cmd, **kwargs):
    print('>', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)

def clone_or_update(path, url):
    if not path.exists():
        run_checked(['git', 'clone', url, path])
    else:
        origin = subprocess.check_output(['git','-C',str(path),'remote','get-url','origin'], text=True).strip()
        assert origin.rstrip('/') == url.rstrip('/'), f'Unexpected repo origin at {path}: {origin}'
        run_checked(['git','-C',path,'pull','--ff-only'])

clone_or_update(GROUP_REPO, GROUP_REPO_URL)
EXPORT_SCRIPT = GROUP_REPO / 'src/inference/export_topk.py'
assert EXPORT_SCRIPT.is_file(), f'Cannot find group exporter: {EXPORT_SCRIPT}'

if not RLMREC_REPO.exists():
    run_checked(['git','clone',RLMREC_URL,RLMREC_REPO])
origin = subprocess.check_output(['git','-C',str(RLMREC_REPO),'remote','get-url','origin'],text=True).strip()
assert origin.rstrip('/') == RLMREC_URL.rstrip('/'), f'RLMRec remote mismatch: {origin}'
current = subprocess.check_output(['git','-C',str(RLMREC_REPO),'rev-parse','HEAD'],text=True).strip()
if current != RLMREC_COMMIT:
    status = subprocess.check_output(['git','-C',str(RLMREC_REPO),'status','--porcelain'],text=True).strip()
    assert not status, 'RLMRec has local edits. Restart the Colab runtime.'
    run_checked(['git','-C',RLMREC_REPO,'fetch','origin'])
    run_checked(['git','-C',RLMREC_REPO,'checkout','--detach',RLMREC_COMMIT])

# Symlink dataset for upstream model, just as E1 notebook did.
link = RLMREC_REPO / 'data/amazon'
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink():
    assert link.resolve() == DATA_DIR.resolve(), f'Unexpected data symlink: {link}'
elif link.exists():
    raise RuntimeError(f'{link} exists and is not a symlink; please inspect manually.')
else:
    link.symlink_to(DATA_DIR, target_is_directory=True)

GROUP_COMMIT = subprocess.check_output(['git','-C',str(GROUP_REPO),'rev-parse','HEAD'],text=True).strip()
EXPORTER_SHA256 = hashlib.sha256(EXPORT_SCRIPT.read_bytes()).hexdigest()
source_text = EXPORT_SCRIPT.read_text(encoding='utf-8')
for flag in ['--root','--model','--seed','--k']:
    assert flag in source_text, f'Exporter lacks expected CLI option {flag}; review source before export.'
# NOTE: In v1 this blanket assertion blocked even a harmless CLI default or comment.
# We now inspect the exact occurrences and isolate an exporter copy per run.
# See prepare_runtime_exporter(stage) in Cell 4.
drive_hits = [(i, ln.strip()) for i, ln in enumerate(source_text.splitlines(), 1)
              if '/content/drive/MyDrive/' in ln]
if drive_hits:
    print('INFO: Group exporter contains Google Drive path literals:')
    for lineno, ln in drive_hits[:20]:
        print(f'  L{lineno}: {ln[:160]}')
    if len(drive_hits) > 20:
        print(f'  ... and {len(drive_hits) - 20} more hits')
    print('INFO: They will be handled safely by the staging root patch in Cell 4.')
else:
    print('INFO: No absolute Google Drive paths detected in exporter source.')
print('Group exporter git SHA:', GROUP_COMMIT)
print('Exporter sha256:', EXPORTER_SHA256)
print('RLMRec source commit:', RLMREC_COMMIT)
print('Exporter source:', EXPORT_SCRIPT)

In [ ]:
# CELL 3 — CHECK DATA & E3C PROTOCOL / CHECKPOINT MATRIX
protocol_file = E3C_ROOT / 'E3C_protocol.json'
if protocol_file.exists():
    protocol = json.loads(protocol_file.read_text(encoding='utf-8'))
    assert protocol.get('model') == MODEL
    assert protocol.get('source_commit') == RLMREC_COMMIT
    assert protocol.get('evaluation', {}).get('ranking_mask') == 'train_only'
    assert protocol.get('evaluation', {}).get('validation_masked') is False
    assert protocol.get('architecture', {}).get('graph_layers') == 3
    assert protocol.get('architecture', {}).get('embedding_dimension') == 32
    assert [float(x) for x in WEIGHTS] == [float(x) for x in protocol['weights']]
    print('E3C protocol: PASS')
else:
    print('WARNING: E3C_protocol.json not found; proceed with notebook pinned protocol.')

with open(DATA_DIR/'trn_mat.pkl','rb') as f:
    TRAIN = pickle.load(f)
with open(DATA_DIR/'tst_mat.pkl','rb') as f:
    TEST = pickle.load(f)
assert sp.issparse(TRAIN) and sp.issparse(TEST)
TRAIN = TRAIN.tocsr(); TEST = TEST.tocsr()
TRAIN.sort_indices(); TEST.sort_indices()
assert TRAIN.shape == TEST.shape == (11000, 9332), (TRAIN.shape, TEST.shape)
TEST_USERS = np.flatnonzero(np.diff(TEST.indptr)>0)
assert len(TEST_USERS) == 9860, len(TEST_USERS)
print('Dataset:', TRAIN.shape, '| test users:', len(TEST_USERS))

PLAN = []
for weight in WEIGHTS:
    for seed in SEEDS:
        run_dir = E3C_ROOT / 'runs/full' / f'weight_{weight}' / f'seed_{seed}'
        ckpt = run_dir/'checkpoints'/f'{MODEL}-{DATASET}-{seed}.pth'
        source = PRED_ROOT/f'seed_{seed}'/'topk_test.npz'
        is_reuse = (weight == REUSED_WEIGHT)
        has_source = source.is_file() if is_reuse else ckpt.is_file()
        PLAN.append({'weight':weight,'seed':seed,'reuse_e1':is_reuse,
                     'run_dir':run_dir, 'checkpoint':ckpt, 'source_topk':source,
                     'ready':has_source, 'topk_dir':run_dir/'topk'})
for task in PLAN:
    print(('REUSE' if task['reuse_e1'] else 'NEW  '),
          f"w={task['weight']:>5} seed={task['seed']}",
          'READY' if task['ready'] else 'MISSING INPUT')
assert all(t['ready'] for t in PLAN), 'Missing E3C checkpoint or E1 Top-K. See plan above.'

In [ ]:
# CELL 4 — EXPORT HELPERS / VERIFICATION / PROVENANCE
from tempfile import mkdtemp
from pprint import pprint

def sha256_file(path, chunk=2**20):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json_atomic(path, payload):
    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2),encoding='utf-8')
    os.replace(tmp, path)

def verify_topk(path, verification_path=None):
    path=Path(path)
    assert path.is_file(), f'Missing {path}'
    with np.load(path,allow_pickle=False) as z:
        assert {'user_ids','item_ids','scores'}.issubset(z.files), z.files
        users=z['user_ids']; items=z['item_ids']; scores=z['scores']
    assert users.ndim==1 and users.shape==(len(TEST_USERS),), users.shape
    assert items.shape==(len(TEST_USERS),K), items.shape
    assert scores.shape==items.shape, scores.shape
    assert np.issubdtype(users.dtype,np.integer) and np.issubdtype(items.dtype,np.integer)
    assert np.array_equal(np.sort(users),TEST_USERS), 'Test-user set mismatch'
    assert len(np.unique(users))==len(users), 'Duplicate test users'
    assert items.min()>=0 and items.max()<TRAIN.shape[1], 'Invalid item IDs'
    assert np.isfinite(scores).all(), 'Non-finite ranking scores'
    assert np.all(scores[:,:-1] >= scores[:,1:] - 1e-6), 'Scores are not descending'
    assert np.all(np.sort(items,axis=1)[:,1:] != np.sort(items,axis=1)[:,:-1]), 'Duplicate items in Top-K'
    # User-by-user CSR check; never exclude validation/test positives from candidates.
    if CHECK_TRAIN_MASK:
        for u, ids in zip(users,items):
            row=TRAIN.indices[TRAIN.indptr[u]:TRAIN.indptr[u+1]]
            if np.isin(ids,row).any():
                raise AssertionError(f'Train leakage for user {u}')
    if verification_path is not None:
        v=load_json(verification_path)
        assert v.get('status')=='passed', f'Exporter verification failed: {v}'
    return {'status':'passed','n_test_users':int(len(users)),'K':int(K),
            'n_items':int(TRAIN.shape[1]), 'train_mask_checked':bool(CHECK_TRAIN_MASK),
            'no_duplicates':True,'scores_descending':True,'scores_finite':True}

def parse_e3c_training_log(run_dir, weight, seed):
    logpath=run_dir/'training_console.log'
    assert logpath.is_file(), f'Missing E3C training log: {logpath}'
    text=logpath.read_text(encoding='utf-8',errors='replace')
    # Each log is from a single training run. Extract its final test result (not E1).
    pat = r"Best Epoch\s+(\d+)\.\s+Final test result:\s*\{'recall': array\(\[([^\]]+)\]\),\s*'ndcg': array\(\[([^\]]+)\]\)\}"
    matches=list(re.finditer(pat,text,re.S))
    assert matches, f'Cannot parse final trainer metrics in {logpath}'
    m=matches[-1]
    extract=lambda s: [float(v) for v in re.findall(r'[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?',s)]
    recalls, ndcgs = extract(m.group(2)),extract(m.group(3))
    assert len(recalls)==len(ndcgs)==3, (recalls,ndcgs)
    metric_values = {**{f'recall@{k}':v for k,v in zip([5,10,20],recalls)},
                     **{f'ndcg@{k}':v for k,v in zip([5,10,20],ndcgs)}}
    # Original training config appears in first console output as Python dictionary.
    training_config=None
    for line in text.splitlines():
        if line.startswith("{'optimizer':"):
            try: training_config=ast.literal_eval(line); break
            except (ValueError,SyntaxError): continue
    assert training_config is not None, f'Cannot parse training config: {logpath}'
    assert training_config['train']['seed']==seed
    assert training_config['model']['name']==MODEL
    assert math.isclose(float(training_config['model']['kd_weight']),float(weight),abs_tol=1e-10)
    assert math.isclose(float(training_config['model'][DATASET]['kd_weight']),float(weight),abs_tol=1e-10)
    assert training_config['model']['layer_num']==3 and training_config['model']['embedding_size']==32
    status_path = run_dir/'train_status.json'
    status = load_json(status_path)
    assert status.get('exit_code')==0 and status.get('status')=='trained_pending_export',status
    assert status.get('seed')==seed and math.isclose(float(status.get('weight')),float(weight),abs_tol=1e-10)
    assert status.get('source_commit')==RLMREC_COMMIT
    assert any(str(run_dir/'checkpoints') in p for p in status.get('checkpoint_files',[]))
    validations=[float(x) for x in re.findall(r'Epoch\s+\d+\s+Validation set\s+\[[^\]]*recall@20:\s*([0-9.eE+-]+)',text)]
    return {'trainer_metrics':metric_values,'train_config':training_config,
            'best_epoch_index':int(m.group(1)),
            'best_validation_recall20':max(validations) if validations else None,
            'train_status':status, 'training_log':str(logpath)}

def stage_e3c_checkpoint(task, parsed):
    # Staging ensures legacy E1 exporter receives only this exact E3C checkpoint.
    stage=Path(mkdtemp(prefix='e3c_export_',dir='/content'))
    stage_model=stage/'experiments/E1'/MODEL/f"seed_{task['seed']}"
    (stage_model/'checkpoints').mkdir(parents=True,exist_ok=True)
    (stage/'data/raw').mkdir(parents=True,exist_ok=True)
    (stage/'data/raw/amazon').symlink_to(DATA_DIR,target_is_directory=True)
    staged_ckpt=stage_model/'checkpoints'/task['checkpoint'].name
    staged_ckpt.symlink_to(task['checkpoint'])
    metrics={'experiment':'E3C_Semantic_Weight_Sensitivity','model':MODEL,
             'dataset':DATASET,'seed':task['seed'],'weight':float(task['weight']),
             'rlmrec_commit':RLMREC_COMMIT,'best_epoch_index':parsed['best_epoch_index'],
             'best_epoch_number':parsed['best_epoch_index']+1,
             'best_validation_recall20':parsed['best_validation_recall20'],
             'test_metrics':parsed['trainer_metrics'],
             'checkpoint':str(staged_ckpt),'status':'completed'}
    run_config={'experiment':'E3C_Semantic_Weight_Sensitivity','model':MODEL,
                'dataset':DATASET,'seed':task['seed'],'weight':float(task['weight']),
                'rlmrec_commit':RLMREC_COMMIT,'training_config':parsed['train_config'],
                'status':'completed'}
    write_json_atomic(stage_model/'metrics.json',metrics)
    write_json_atomic(stage_model/'run_config.json',run_config)
    # Do not touch any actual E1 files; --root points exclusively to /content stage.
    return stage

def prepare_runtime_exporter(stage):
    """Create a private runtime copy; never edit the source repo or E1 on Drive.

    Hardcoded project-root references in the group exporter are rewritten to
    the per-run staging directory. Explicit --root still points to this stage.
    Non-project absolute Drive paths (if any) fail closed, not guessed over.
    """
    stage = Path(stage)
    source = EXPORT_SCRIPT.read_text(encoding='utf-8')
    assert sha256_file(EXPORT_SCRIPT) == EXPORTER_SHA256, 'Group exporter source changed since Cell 2.'

    old_root = str(ROOT)  # /content/drive/MyDrive/Data_SNA_Group14
    new_root = str(stage)
    patched = source.replace(old_root, new_root)
    replaced_count = source.count(old_root)

    # Fail closed if any OTHER absolute Google Drive path survives in executable
    # source. A comment with a Drive example is harmless and can be retained.
    unexpected = [(i, line.strip()) for i, line in enumerate(patched.splitlines(), 1)
                  if '/content/drive/MyDrive/' in line and not line.lstrip().startswith('#')]
    if unexpected:
        first = '\n'.join(f'  line {i}: {line[:180]}' for i, line in unexpected[:12])
        raise RuntimeError(
            'Exporter contains Google Drive path(s) outside the known project ROOT. '
            'Refusing to run to avoid reading/writing another experiment. '\
            + 'Unpatched references:\n' + first
        )

    # Preserve the original __file__ and import search paths by wrapping exec.
    # Running the exporter as a standalone script previously worked for E1.
    runtime_copy = stage / '_e3c_runtime_export_topk.py'
    compile(patched, str(EXPORT_SCRIPT), 'exec')
    runtime_copy.write_text(patched, encoding='utf-8')
    # Execute with __file__ and sys.path matching the ORIGINAL group exporter:
    # downstream relative filesystem lookups should behave exactly like E1.
    launcher = stage / '_e3c_launch_exporter.py'
    launcher.write_text(
        'import sys\n'
        'from pathlib import Path\n'
        f'original = Path({str(EXPORT_SCRIPT)!r})\n'
        f'patched_path = Path({str(runtime_copy)!r})\n'
        'sys.argv[0] = str(original)\n'
        'sys.path.insert(0, str(original.parent))\n'
        'namespace = {"__name__":"__main__", "__file__":str(original), "__package__":None}\n'
        'exec(compile(patched_path.read_text(encoding="utf-8"), str(original), "exec"), namespace)\n',
        encoding='utf-8'
    )
    note = {'project_root_replaced':old_root,
            'isolated_root':new_root,
            'root_literal_replacements':replaced_count,
            'source_sha256':EXPORTER_SHA256,
            'runtime_source_sha256':sha256_file(runtime_copy),
            'group_commit':GROUP_COMMIT}
    print(f'Exporter isolated: {replaced_count} ROOT literal replacement(s)')
    return launcher, note


def metric_consistency(verification_file, expected):
    v=load_json(verification_file)
    exported=v.get('test_metrics_from_export',{})
    assert exported, 'Exporter did not record test_metrics_from_export; manual accuracy audit required.'
    diffs={}
    for metric in ('recall@10','recall@20','ndcg@10','ndcg@20'):
        if metric in exported:
            diffs[metric]=abs(float(exported[metric])-float(expected[metric]))
    assert {'recall@20','ndcg@20'}.issubset(diffs), f'Missing key metrics in {verification_file}'
    assert max(diffs.values())<=MAX_METRIC_ABS_DIFF, f'Export vs trainer mismatch: {diffs}'
    return diffs

def copy_2phase(src,dst):
    dst=Path(dst);dst.parent.mkdir(parents=True,exist_ok=True)
    tmp=dst.with_name(dst.name+'.partial')
    shutil.copy2(src,tmp)
    os.replace(tmp,dst)

def verified_existing(task, expected_ckpt_sha=None):
    out=task['topk_dir']; provenance=out/'E3C_export_provenance.json'
    if not (out/'topk_test.npz').is_file() or not provenance.is_file():
        return False
    p=load_json(provenance)
    if p.get('weight')!=task['weight'] or p.get('seed')!=task['seed'] or p.get('status')!='passed':
        return False
    if expected_ckpt_sha and p.get('checkpoint_sha256')!=expected_ckpt_sha:
        return False
    try:
        verify_topk(out/'topk_test.npz', out/'export_verification.json')
        return True
    except Exception as e:
        print('Existing output invalid:',e)
        return False

def finalize_run(task, tmp_export_dir, provenance):
    out=task['topk_dir']
    out.mkdir(parents=True,exist_ok=True)
    for name in ['topk_test.npz','export_meta.json','export_verification.json']:
        src=tmp_export_dir/name
        assert src.is_file(), f'Exporter did not produce {src}'
        if name=='export_meta.json':
            raw=load_json(src)
            annotated=dict(raw)
            for k in ['checkpoint','checkpoint_path']:
                if k in annotated:
                    annotated['staged_'+k]=annotated[k]
                    annotated[k]=provenance.get('checkpoint')
            annotated.update({'experiment':'E3C_Semantic_Weight_Sensitivity',
                              'weight':float(task['weight']),
                              'seed':task['seed'],
                              'source_run_dir':str(task['run_dir']),
                              'source_checkpoint_sha256':provenance.get('checkpoint_sha256'),
                              'reference_reuse':task['reuse_e1']})
            write_json_atomic(out/name,annotated)
            copy_2phase(src,out/'export_meta_original.json')
        else:
            copy_2phase(src,out/name)
    # Provenance is written last: marker for safe resume.
    write_json_atomic(out/'E3C_export_provenance.json',provenance)
    return out

In [ ]:
# CELL 5 — EXECUTE 12 EXPORTS + OPTIONAL 3 E1 REUSE
STATUS_PATH = E3C_ROOT/'E3C_export_status.csv'
status_records=[]
new_exports=0

def save_status_table():
    with open(STATUS_PATH.with_suffix('.csv.tmp'),'w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=['weight','seed','mode','status','output','error','updated_utc'])
        writer.writeheader(); writer.writerows(status_records)
    os.replace(STATUS_PATH.with_suffix('.csv.tmp'),STATUS_PATH)

for i,task in enumerate(PLAN,1):
    weight, seed = task['weight'],task['seed']
    reuse = task['reuse_e1']
    mode='reuse_E1' if reuse else 'checkpoint_export'
    print('\n'+'='*74)
    print(f'[{i}/{len(PLAN)}] E3C weight={weight} seed={seed} mode={mode}')
    rec={'weight':weight,'seed':seed,'mode':mode,'status':'pending',
         'output':str(task['topk_dir']),'error':'',
         'updated_utc':datetime.now(timezone.utc).isoformat()}
    stage=None
    try:
        if reuse:
            if not COPY_E1_REFERENCE:
                rec['status']='reference_E1_only'
            else:
                source_dir = task['source_topk'].parent
                source_verify=source_dir/'export_verification.json'
                source_meta=source_dir/'export_meta.json'
                assert source_verify.is_file() and source_meta.is_file(), f'E1 source metadata missing: {source_dir}'
                verify_topk(task['source_topk'],source_verify)
                reused_sha=sha256_file(task['source_topk'])
                if RESUME and not FORCE and verified_existing(task):
                    p=load_json(task['topk_dir']/'E3C_export_provenance.json')
                    assert p.get('source_topk_sha256')==reused_sha, 'Existing reuse source changed; use FORCE=True.'
                    rec['status']='already_verified'
                else:
                    assert FORCE or not (task['topk_dir']/'E3C_export_provenance.json').exists(), \
                        'Existing output differs; review before FORCE=True.'
                    p={'experiment':'E3C_Semantic_Weight_Sensitivity','model':MODEL,
                       'weight':weight,'seed':seed,'mode':'reuse_E1',
                       'source_topk':str(task['source_topk']),
                       'source_topk_sha256':reused_sha,
                       'checkpoint':None,'checkpoint_sha256':None,
                       'group_exporter_commit':GROUP_COMMIT,
                       'validation_masked':False,'ranking_mask':'train_only',
                       'status':'passed','created_utc':datetime.now(timezone.utc).isoformat()}
                    finalize_run(task,source_dir,p)
                    rec['status']='reused_E1_passed'
        else:
            if MAX_NEW_EXPORTS is not None and new_exports>=MAX_NEW_EXPORTS:
                rec['status']='not_run_limit'
            else:
                ckpt=task['checkpoint']
                assert ckpt.is_file() and ckpt.stat().st_size>1_000_000
                ckpt_sha=sha256_file(ckpt)
                if RESUME and not FORCE and verified_existing(task,ckpt_sha):
                    rec['status']='already_verified'
                else:
                    assert FORCE or not (task['topk_dir']/'E3C_export_provenance.json').exists(), \
                        'Existing output marker conflicts: review before FORCE=True.'
                    parsed=parse_e3c_training_log(task['run_dir'],weight,seed)
                    assert parsed['train_status'].get('checkpoint_bytes')==ckpt.stat().st_size
                    stage=stage_e3c_checkpoint(task,parsed)
                    runtime_exporter, runtime_patch = prepare_runtime_exporter(stage)
                    cmd=[sys.executable,str(runtime_exporter),
                         '--repo',str(RLMREC_REPO),'--root',str(stage),
                         '--model',MODEL,'--seed',str(seed),'--k',str(K)]
                    logpath=task['run_dir']/'E3C_export_console.log'
                    print('E3C checkpoint:',ckpt)
                    print('Stage root (NOT E1):',stage)
                    print('Command:', ' '.join(cmd))
                    export_env = os.environ.copy()
                    import_paths = [str(EXPORT_SCRIPT.parent), str(GROUP_REPO)]
                    if export_env.get('PYTHONPATH'):
                        import_paths.append(export_env['PYTHONPATH'])
                    export_env['PYTHONPATH'] = os.pathsep.join(import_paths)
                    with open(logpath,'w',encoding='utf-8') as f:
                        process=subprocess.run(cmd,stdout=f,stderr=subprocess.STDOUT,
                                               text=True,env=export_env)
                    if process.returncode!=0:
                        tail='\n'.join(logpath.read_text(encoding='utf-8',errors='replace').splitlines()[-60:])
                        raise RuntimeError(f'Exporter exit code {process.returncode}. Log tail:\n{tail}')
                    temp_out=stage/'experiments/predictions'/MODEL/f'seed_{seed}'
                    # Require output *in stage*, not in E1 Drive path. Also check
                    # any checkpoint metadata refers to this same E3C checkpoint.
                    export_meta_file = temp_out/'export_meta.json'
                    assert export_meta_file.is_file(), f'Missing staged exporter metadata: {export_meta_file}'
                    original_meta = load_json(export_meta_file)
                    for field in ['checkpoint', 'checkpoint_path', 'checkpoint_file']:
                        recorded = original_meta.get(field)
                        if isinstance(recorded,str) and recorded.endswith('.pth'):
                            got_path = Path(recorded)
                            if not got_path.is_absolute():
                                got_path = (stage / got_path)
                            assert got_path.resolve() == ckpt.resolve(), (field, recorded, ckpt)
                    upstream_v=temp_out/'export_verification.json'
                    audit=verify_topk(temp_out/'topk_test.npz',upstream_v)
                    metric_diffs=metric_consistency(upstream_v,parsed['trainer_metrics'])
                    provenance={'experiment':'E3C_Semantic_Weight_Sensitivity',
                                'model':MODEL,'dataset':DATASET,'weight':weight,'seed':seed,
                                'mode':'checkpoint_export','status':'passed',
                                'checkpoint':str(ckpt),'checkpoint_sha256':ckpt_sha,
                                'staging_root':str(stage),
                                'group_exporter_commit':GROUP_COMMIT,
                                'group_exporter_sha256':EXPORTER_SHA256,
                                'runtime_exporter_patch':runtime_patch,
                                'rlmrec_commit':RLMREC_COMMIT,
                                'ranking_mask':'train_only','validation_masked':False,
                                'trainer_metrics_from_E3C_log':parsed['trainer_metrics'],
                                'export_vs_trainer_abs_diff':metric_diffs,
                                'audit':audit,'created_utc':datetime.now(timezone.utc).isoformat()}
                    out=finalize_run(task,temp_out,provenance)
                    assert sha256_file(ckpt)==ckpt_sha, 'Source checkpoint changed during export'
                    verify_topk(out/'topk_test.npz',out/'export_verification.json')
                    rec['status']='export_passed'
                    new_exports+=1
                    print('PASS:', out)
    except Exception as e:
        rec['status']='FAILED';rec['error']=str(e)[:1000]
        print('FAILED:',rec['error'])
        traceback.print_exc(limit=2)
    finally:
        if stage is not None and stage.exists():
            shutil.rmtree(stage)  # local /content temporary staging only
        rec['updated_utc']=datetime.now(timezone.utc).isoformat()
        status_records.append(rec)
        save_status_table()
    if rec['status']=='FAILED' and STOP_ON_ERROR:
        raise RuntimeError(f"Stopped at weight={weight}, seed={seed}. Check {task['run_dir']/'E3C_export_console.log'}")
print('\nStatus file:',STATUS_PATH)

In [ ]:
# CELL 6 — SUMMARY / FINAL AUDIT (run after Cell 5)
import pandas as pd
status_df=pd.read_csv(STATUS_PATH)
print(status_df[['weight','seed','mode','status']].to_string(index=False))
print('\nStatus counts:')
print(status_df['status'].value_counts(dropna=False).to_string())

all_out=[]
for task in PLAN:
    p=task['topk_dir']/'E3C_export_provenance.json'
    if p.is_file() and (task['topk_dir']/'topk_test.npz').is_file():
        prov=load_json(p)
        if prov.get('status')=='passed':
            all_out.append((task['weight'],task['seed'],str(task['topk_dir'])))
print(f'\nPASS outputs found: {len(all_out)} / 15')
for weight, seed, out in all_out:
    print(f'  weight={weight:>5} seed={seed}: {out}')
print('\nOutput convention: experiments/E3C/runs/full/weight_<w>/seed_<s>/topk/')
print('Main file: topk_test.npz; auxiliary: export_meta.json, export_verification.json, E3C_export_provenance.json')
print('Status CSV:', STATUS_PATH)
assert len(all_out)==15 or MAX_NEW_EXPORTS is not None or not COPY_E1_REFERENCE, \
    'Not all 15 outputs are verified — check status CSV before E3C bias analysis.'